# Attention, one step at a time


**The road**

| Part | What we add | Why |
|---|---|---|
| 1 | dot-product scores | a first way to compare tokens |
| 2 | Q, K, V | make the comparison *learnable* |
| 3 | `/ sqrt(d_k)` | keep softmax from saturating |
| 4 | causal mask | stop the model from cheating |
| 5 | softmax @ V | actually mix the tokens |
| 6 | **function:** one head | first checkpoint |
| 7 | many heads | one head is a compromise |
| 8 | **function:** many heads | second checkpoint |
| 9 | KV cache | make generation fast |
| 10 | **class:** `MultiHeadAttention` | the real file |

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=2, sci_mode=False, linewidth=120)

print("torch", torch.__version__)

torch 2.13.0+cpu


In [6]:
def show(mat, title="", row_labels=None, col_labels=None, decimals=2):
    """Print a 2-D tensor with token names on the rows and columns."""
    row_labels = row_labels or TOKENS
    col_labels = col_labels or TOKENS
    if title:
        print(title)
    print("       " + "".join(f"{c:>8}" for c in col_labels))
    for name, row in zip(row_labels, mat.tolist()):
        body = "".join(f"{val:8.{decimals}f}" for val in row)
        print(f"{name:>6} {body}")
    print()


In [23]:
from IPython.display import HTML, display

BLUE, RED, GREEN, AMBER = "#2563eb", "#e11d48", "#059669", "#d97706"
HEAD_COLORS = [BLUE, RED, GREEN, AMBER, "#7c3aed", "#0891b2"]

# No quotes inside these: they are interpolated into single-quoted HTML/SVG
# attributes, and a nested quote would close the attribute early. CSS accepts
# multi-word family names unquoted.
FONT  = "system-ui,-apple-system,Segoe UI,sans-serif"
MONO  = "ui-monospace,Consolas,Menlo,monospace"


def _mix(hex_color, t):
    """t = 0 -> white,  t = 1 -> the full colour."""
    t = max(0.0, min(1.0, t))
    r, g, b = (int(hex_color[i:i + 2], 16) for i in (1, 3, 5))
    return "rgb(%d,%d,%d)" % (round(255 + (r - 255) * t),
                              round(255 + (g - 255) * t),
                              round(255 + (b - 255) * t))


def heatmap(mat, title="", row_labels=None, col_labels=None,
            color=BLUE, fmt="{:.2f}", vmin=None, vmax=None):
    """A 2-D tensor as a colour-shaded HTML table. Returns an HTML string."""
    vals = mat.detach().tolist() if hasattr(mat, "detach") else [list(r) for r in mat]

    def names(k):                       # falls back to indices past the toy sentence
        return [TOKENS[i] if i < len(TOKENS) else str(i) for i in range(k)]

    row_labels = row_labels or names(len(vals))
    col_labels = col_labels or names(len(vals[0]))
    finite = [v for r in vals for v in r if float("-inf") < v < float("inf")]
    lo = min(finite) if vmin is None else vmin
    hi = max(finite) if vmax is None else vmax
    span = (hi - lo) or 1.0

    th = f"font:600 11px {FONT};color:#64748b;padding:2px 6px"
    td = f"font:600 11px {MONO};padding:4px 7px;text-align:center;border-radius:3px"

    out = ["<div style='display:inline-block;margin:0 16px 12px 0'>"]
    if title:
        out.append(f"<div style='font:650 12px {FONT};color:#334155;"
                   f"margin-bottom:6px'>{title}</div>")
    out.append("<table style='border-collapse:separate;border-spacing:2px'><tr><td></td>")
    out += [f"<th style='{th};text-align:center'>{c}</th>" for c in col_labels]
    out.append("</tr>")
    for name, row in zip(row_labels, vals):
        out.append(f"<tr><th style='{th};text-align:right'>{name}</th>")
        for v in row:
            if v == float("-inf"):
                out.append(f"<td style='{td};background:#f8fafc;color:#cbd5e1;"
                           f"border:1px dashed #cbd5e1'>-inf</td>")
            else:
                t = (v - lo) / span
                fg = "#fff" if t > 0.62 else "#1e293b"
                out.append(f"<td style='{td};background:{_mix(color, t)};"
                           f"color:{fg}'>{fmt.format(v)}</td>")
        out.append("</tr>")
    out.append("</table></div>")
    return "".join(out)


def figure(*parts, caption=""):
    """Lay several HTML blocks side by side and display them."""
    body = ("<div style='display:flex;flex-wrap:wrap;align-items:flex-start'>"
            + "".join(parts) + "</div>")
    if caption:
        body += (f"<div style='font:12px {FONT};color:#64748b;max-width:760px;"
                 f"margin-top:4px'>{caption}</div>")
    display(HTML(body))


def linechart(xs, series, title="", xlabel="", ylabel="",
              width=430, height=250, xticklabels=None, yfmt="{:g}"):
    """A line chart in plain SVG.  series = [(label, ys, colour), ...]"""
    pl, pr, pt, pb = 58, 14, 30, 42
    allv = [v for _, ys, _ in series for v in ys]
    lo, hi = min(allv + [0]), max(allv)
    if hi == lo:
        hi = lo + 1
    idx = list(range(len(xs)))
    xvals = idx if xticklabels else xs
    x0, x1 = min(xvals), max(xvals)

    def px(x):
        return pl + (x - x0) / ((x1 - x0) or 1) * (width - pl - pr)

    def py(y):
        return height - pb - (y - lo) / ((hi - lo) or 1) * (height - pb - pt)

    s = [f"<svg width='{width}' height='{height}' style='overflow:visible'>"]
    if title:
        s.append(f"<text x='{pl}' y='16' font-family='{FONT}' font-size='12.5' "
                 f"font-weight='650' fill='#334155'>{title}</text>")

    for i in range(5):                                     # grid + y labels
        y = lo + (hi - lo) * i / 4
        yy = py(y)
        s.append(f"<line x1='{pl}' y1='{yy:.1f}' x2='{width - pr}' y2='{yy:.1f}' "
                 f"stroke='#e2e8f0' stroke-width='1'/>")
        s.append(f"<text x='{pl - 8}' y='{yy + 3.5:.1f}' text-anchor='end' "
                 f"font-family='{MONO}' font-size='10' fill='#94a3b8'>"
                 f"{yfmt.format(y)}</text>")

    labels = xticklabels or [f"{v:g}" for v in xs]
    step = max(1, len(labels) // 8)
    for i, lab in enumerate(labels):
        if i % step and i != len(labels) - 1:
            continue
        xx = px(xvals[i])
        s.append(f"<text x='{xx:.1f}' y='{height - pb + 15}' text-anchor='middle' "
                 f"font-family='{MONO}' font-size='10' fill='#94a3b8'>{lab}</text>")

    for j, (label, ys, colour) in enumerate(series):       # the lines
        pts = " ".join(f"{px(xvals[i]):.1f},{py(v):.1f}" for i, v in enumerate(ys))
        s.append(f"<polyline points='{pts}' fill='none' stroke='{colour}' "
                 f"stroke-width='2.2' stroke-linejoin='round'/>")
        if len(ys) <= 24:
            for i, v in enumerate(ys):
                s.append(f"<circle cx='{px(xvals[i]):.1f}' cy='{py(v):.1f}' r='3' "
                         f"fill='#fff' stroke='{colour}' stroke-width='2'/>")
        lx = width - pr - 8
        s.append(f"<text x='{lx}' y='{20 + j * 15}' text-anchor='end' "
                 f"font-family='{FONT}' font-size='11' font-weight='650' "
                 f"fill='{colour}'>{label}</text>")

    if xlabel:
        s.append(f"<text x='{(pl + width - pr) / 2:.0f}' y='{height - 6}' "
                 f"text-anchor='middle' font-family='{FONT}' font-size='11' "
                 f"fill='#64748b'>{xlabel}</text>")
    if ylabel:
        s.append(f"<text transform='translate(13,{(pt + height - pb) / 2:.0f}) rotate(-90)' "
                 f"text-anchor='middle' font-family='{FONT}' font-size='11' "
                 f"fill='#64748b'>{ylabel}</text>")
    s.append("</svg>")
    return "<div style='margin:0 18px 10px 0'>" + "".join(s) + "</div>"


print("drawing helpers ready")

drawing helpers ready


## A toy sentence

The real model uses `d_model=768` and `n_heads=12`. We cannot read a 768-wide
matrix, so this notebook uses **deliberately tiny numbers**:

```
B       = 1     one sentence in the batch
T       = 5     five tokens
d_model = 8     each token is an 8-number vector
n_heads = 2     two heads
d_k     = 4     8 / 2 -> each head gets 4 numbers
```

Every shape you see below is the real shape, just small.

In [ ]:
TOKENS  = ["the", "cat", "sat", "on", "mat"]

B       = 1
T       = len(TOKENS)
d_model = 8
n_heads = 2
d_k     = d_model // n_heads

x = torch.randn(B, T, d_model)

print("tokens :", TOKENS)
print("x.shape:", tuple(x.shape), " (batch, time, width)")
print()
print("x[0] -- one row per token:")
print(x)

tokens : ['the', 'cat', 'sat', 'on', 'mat']
x.shape: (1, 5, 8)  (batch, time, width)

x[0] -- one row per token:
tensor([[[ 1.17,  0.77,  0.39,  0.29, -2.76, -0.83,  0.49,  0.29],
         [-1.13, -0.00, -0.16, -0.25,  2.42,  1.65, -0.31, -1.51],
         [-0.56, -0.83, -1.40, -0.40, -0.31, -0.06,  0.52, -1.60],
         [ 0.50,  0.88,  0.39,  1.46,  0.48, -0.53, -0.03,  0.66],
         [-0.31, -0.56, -0.48, -1.27, -0.17,  0.55, -0.18, -0.23]]])


In [10]:
col_labels = [ f"d{i}" for i in range(5) ]
show(x[0][:, :5], "x, first 5 columns", col_labels=col_labels)

x, first 5 columns
             d0      d1      d2      d3      d4
   the     1.17    0.77    0.39    0.29   -2.76
   cat    -1.13   -0.00   -0.16   -0.25    2.42
   sat    -0.56   -0.83   -1.40   -0.40   -0.31
    on     0.50    0.88    0.39    1.46    0.48
   mat    -0.31   -0.56   -0.48   -1.27   -0.17



## compare tokens with a dot product

Right now every token vector knows **only its own word**. `x[2]` is "sat" and has
no idea that a cat is involved.

We want each token to build a *new* vector by mixing in the tokens it cares about:

```
new_sat = 0.6 * cat  +  0.3 * sat  +  0.1 * the
```

So the question is: **where do the numbers 0.6 / 0.3 / 0.1 come from?**

First attempt: two vectors that point in a similar direction have a large dot
product. So let us just compare every token with every other token.

In [13]:
naive_scores = x[0] @ x[0].T

print("x[0].shape:", tuple(x[0].shape))
print("naive_scores.shape:", tuple(naive_scores.shape))

show(naive_scores, "naive_scores[i][j] = how much token i matches token j")

x[0].shape: (5, 8)
naive_scores.shape: (5, 5)
naive_scores[i][j] = how much token i matches token j
            the     cat     sat      on     mat
   the    10.82  -10.09   -1.27    1.13   -1.49
   cat   -10.09   12.32    2.38   -1.69    1.65
   sat    -1.27    2.38    6.03   -3.33    2.13
    on     1.13   -1.69   -3.33    4.26   -3.22
   mat    -1.49    1.65    2.13   -3.22    2.69



---
## Q, K, V: give each token three roles

Instead of comparing `x` with `x`, we let the model learn **three different
projections** of every token.

| Name | The token is asking / offering | Library analogy |
|---|---|---|
| **Q**uery | "what am I looking for?" | what you want to read |
| **K**ey   | "what do I advertise?"   | the title on the book's spine |
| **V**alue | "what do I hand over if picked?" | the content inside the book |

You search with your **query**, you match against **keys**, and you walk away
with **values**.

Each is one `nn.Linear`. In the real file:

```python
self.W_q = nn.Linear(config.d_model, config.d_model, bias=config.qkv_bias)
```

`qkv_bias` is `False` in `config.py`: a bias shifts Q/K/V but changes nothing
about what attention computes, so it is left out.

In [52]:
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)
W_o = nn.Linear(d_model, d_model)

q = W_q(x)
k = W_k(x)
v = W_v(x)

print("x.shape:", tuple(x.shape))
print("q.shape:", tuple(q.shape))
print("k.shape:", tuple(k.shape))
print("v.shape:", tuple(v.shape))
print()

x.shape: (1, 5, 8)
q.shape: (1, 5, 8)
k.shape: (1, 5, 8)
v.shape: (1, 5, 8)



In [ ]:
print(x[0][2])
print(q[0][2])
print(k[0][2])

tensor([-0.52, -0.06, -0.33,  0.17, -0.32,  0.06,  0.79,  0.33], grad_fn=<SelectBackward0>)


Now compare **queries against keys** instead of `x` against `x`.

`k.transpose(-2, -1)` swaps the last two axes: `(B, T, d_model) -> (B, d_model, T)`.


We use `-2, -1` rather than `1, 2` so the same line keeps working later, when a
head axis appears in front.

In [21]:
scores = q @ k.transpose(-2, -1)

print("scores.shape:", tuple(scores.shape))
print()
show(scores[0], "scores = q @ k^T")

scores.shape: (1, 5, 5)

scores = q @ k^T
            the     cat     sat      on     mat
   the     1.06    0.09    0.37   -0.44    0.46
   cat    -2.25    1.27    0.71   -0.01   -0.13
   sat    -0.94    0.96   -0.32    0.18   -0.01
    on     0.82   -0.95   -0.30    0.19   -0.13
   mat    -0.41    0.27    0.24   -0.07   -0.01



---
## Scale by `sqrt(d_k)`

We are about to push these scores through a softmax. A dot product over many
dimensions produces large numbers, and softmax of large numbers collapses into a
single winner.

Let us see that happen.

In [ ]:
def row(t, style="{:6.2f}"):
    return "[" + "  ".join(style.format(v) for v in t.tolist()) + "]"

small = torch.tensor([0.5, 1.2, 0.8])
big = small * 16

print("scores", row(small), "-> softmax", row(F.softmax(small, dim=-1), "{:6.1%}"))
print("scores", row(big),   "-> softmax", row(F.softmax(big,   dim=-1), "{:6.1%}"))

scores [  0.50    1.20    0.80] -> softmax [ 22.9%   46.1%   30.9%]
scores [  8.00   19.20   12.80] -> softmax [  0.0%   99.8%    0.2%]


In [25]:
scaled = scores / d_k ** 0.5

print("std of raw scores   :", round(scores.std().item(), 3))
print("std of scaled scores:", round(scaled.std().item(), 3))
print()
show(scaled[0], "scaled = scores / sqrt(d_k)")

std of raw scores   : 0.73
std of scaled scores: 0.365

scaled = scores / sqrt(d_k)
            the     cat     sat      on     mat
   the     0.53    0.05    0.19   -0.22    0.23
   cat    -1.12    0.64    0.35   -0.00   -0.07
   sat    -0.47    0.48   -0.16    0.09   -0.01
    on     0.41   -0.47   -0.15    0.10   -0.06
   mat    -0.20    0.13    0.12   -0.03   -0.01



---
## The causal mask: no peeking at the future

Our model's job is to predict the **next** token. If, while processing "sat", the
model could look at "mat", it would be reading the answer off the page. At
generation time the future does not exist, so a model trained that way would
collapse.

So we forbid every token from seeing anything to its right.

`torch.tril` ("**tri**angle, **l**ower") builds exactly that pattern.

In [ ]:
# example

max_seq_len = 16

sent = torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)
sent_mask = torch.tril(sent)

print("sent.shape:", tuple(sent.shape))
print("mask.shape:", tuple(sent_mask.shape))

show(sent[:T, :T].int())
print()
show(sent_mask[:T, :T].int())

sent.shape: (16, 16)
mask.shape: (16, 16)
            the     cat     sat      on     mat
   the     1.00    1.00    1.00    1.00    1.00
   cat     1.00    1.00    1.00    1.00    1.00
   sat     1.00    1.00    1.00    1.00    1.00
    on     1.00    1.00    1.00    1.00    1.00
   mat     1.00    1.00    1.00    1.00    1.00


            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     1.00    1.00    0.00    0.00    0.00
   sat     1.00    1.00    1.00    0.00    0.00
    on     1.00    1.00    1.00    1.00    0.00
   mat     1.00    1.00    1.00    1.00    1.00



In [32]:
# Let's apply

mask = torch.tril(
    torch.ones(T, T, dtype=torch.bool)
)

masked = scaled.masked_fill(~mask, float("-inf"))

show(masked[0], "scaled scores after masked_fill")

scaled scores after masked_fill
            the     cat     sat      on     mat
   the     0.53    -inf    -inf    -inf    -inf
   cat    -1.12    0.64    -inf    -inf    -inf
   sat    -0.47    0.48   -0.16    -inf    -inf
    on     0.41   -0.47   -0.15    0.10    -inf
   mat    -0.20    0.13    0.12   -0.03   -0.01



---
# Softmax, then mix the values

`dim=-1` is important: we normalise **across the keys**, i.e. along each row.
"Given this query, how do I spread 100% of my attention over the tokens I am
allowed to see?"

In [35]:
weights = F.softmax(masked, dim=-1) # (B, T, T)
out = weights @ v # (B, T, T) @ (B, T, d_model) => (B, T, d_model)

print("weights.shape:", tuple(weights.shape))
print("v.shape      :", tuple(v.shape))
print("out.shape    :", tuple(out.shape), " <- same shape we started with")
print()
show(weights[0], "attention weights (each row sums to 1)")

weights.shape: (1, 5, 5)
v.shape      : (1, 5, 8)
out.shape    : (1, 5, 8)  <- same shape we started with

attention weights (each row sums to 1)
            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     0.15    0.85    0.00    0.00    0.00
   sat     0.20    0.52    0.28    0.00    0.00
    on     0.37    0.15    0.21    0.27    0.00
   mat     0.16    0.23    0.22    0.19    0.20



---
# Checkpoint: wrap one head in a function

Nothing new below. We are only collecting Parts 2-5 into one place.

In [39]:
def single_head_attention(x, W_q, W_k, W_v, mask):
    """One attention head.

    x -> (B, T, d_model);  returns (B, T, d_model) and the attention weights.
    """

    B, T, d_model = x.shape

    q = W_q(x)                                          # Part 2
    k = W_k(x)
    v = W_v(x)

    scores = q @ k.transpose(-2, -1)                    # Part 2
    scores = scores / d_model**0.5                      # Part 3

    causal = mask[:T, :T]                               # Part 4
    scores = scores.masked_fill(~causal, float("-inf"))

    weights = F.softmax(scores, dim=-1)
    out = weights @ v

    return out, weights

In [41]:
head_out, head_w = single_head_attention(x, W_q, W_k, W_v, mask)

print("in :", tuple(x.shape))
print("out:", tuple(head_out.shape))

in : (1, 5, 8)
out: (1, 5, 8)


---
# One head is not enough

A sentence contains many kinds of relation at the same time:

* subject -> verb        ("cat" -> "sat")
* adjective -> noun
* pronoun -> what it refers to

One head produces **one** `(T, T)` weight matrix, so it has to compromise between
all of them.

Are two heads really different, or do they learn the same thing? Let us test it:
build two heads with independent random weights and compare their patterns.

In [42]:
head_a, head_b = None, None

torch.manual_seed(101)
head_a = [
    nn.Linear(d_model, d_model, bias=False)
    for _ in range(3)
]

torch.manual_seed(202)
head_b = [
    nn.Linear(d_model, d_model, bias=False)
    for _ in range(3)
]

o_a, w_a = single_head_attention(x, *head_a, mask)
o_b, w_b = single_head_attention(x, *head_b, mask)

show(w_a[0], "head A")
show(w_b[0], "head B")

head A
            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     0.79    0.21    0.00    0.00    0.00
   sat     0.27    0.43    0.30    0.00    0.00
    on     0.19    0.28    0.33    0.21    0.00
   mat     0.26    0.17    0.15    0.23    0.19

head B
            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     0.68    0.32    0.00    0.00    0.00
   sat     0.31    0.31    0.38    0.00    0.00
    on     0.25    0.29    0.23    0.24    0.00
   mat     0.21    0.17    0.20    0.21    0.21



## How to run many heads cheaply

**The obvious way:** make 12 small `nn.Linear(768, 64)` layers and loop over them.
Correct, but that is 12 separate small matmuls -- slow on a GPU.

**The trick used in the file:** keep *one* full-width `nn.Linear(768, 768)` per
role, and **carve** the heads out of its output by reshaping.

```
(B, T, 8)                         after W_q
   | .view(B, T, 2, 4)            split the width into 2 chunks of 4
(B, T, 2, 4)
   | .transpose(1, 2)             move the head axis in front of time
(B, 2, T, 4)                      now: 2 independent (T, 4) problems
```

In [49]:
# projected = W_q(x)
# print("1. after W_q            :", tuple(projected.shape))

# split = projected.view(B, T, n_heads, d_k)
# print("2. after .view          :", tuple(split.shape), " (B, T, n_heads, d_k)")

# q_heads = split.transpose(1, 2)
# print("3. after .transpose(1,2):", tuple(q_heads.shape), " (B, n_heads, T, d_k)")

q_heads = W_q(x).view(B, T, n_heads, d_k).transpose(1, 2)
print("q_heads:", tuple(q_heads.shape), " (B, n_heads, T, d_k)")

q_heads: (1, 2, 5, 4)  (B, n_heads, T, d_k)


## Checkpoint: wrap multi-head attention in a function

In [51]:
def multi_head_attention(x, W_q, W_k, W_v, W_o, mask, n_heads):
    """Multi-head causal self-attention.  (B, T, d_model) -> (B, T, d_model)"""

    B, T, d_model = x.shape
    d_k = d_model // n_heads

    q = W_q(x).view(B, T, n_heads, d_k).transpose(1, 2)
    k = W_k(x).view(B, T, n_heads, d_k).transpose(1, 2) # => (B, n_heads, T, d_k)
    v = W_v(x).view(B, T, n_heads, d_k).transpose(1, 2)

    scores = q @ k.transpose(-2, -1)
    scores = scores / d_k**0.5
    scores = scores.masked_fill(~mask, float("-inf"))

    weights = F.softmax(scores, dim=-1)
    out = weights @ v

    out = out.transpose(1, 2).contiguous().view(B, T, d_model)
    # .transpose(1, 2) ===> (B, T, n_heads, d_k)
    # .view(B, T, d_model) ===> (B, T, d_model)

    return W_o(out), weights


In [53]:

mha_out, mha_w =  multi_head_attention(x, W_q, W_k, W_v, W_o, mask, n_heads)


print("in :", tuple(x.shape))
print("out:", tuple(mha_out.shape))
print("weights:", tuple(mha_w.shape), " (B, n_heads, T, T)")

in : (1, 5, 8)
out: (1, 5, 8)
weights: (1, 2, 5, 5)  (B, n_heads, T, T)


---

## The MyMultiHeadAttention

In [ ]:
class MyMultiHeadAttention(nn.Module):
    """Every token looks at every earlier token, through several lenses at once.

    Input:  (B, T, d_model)
    Output: (B, T, d_model)
    """

    def __init__(self, config):
        super().__init__()

        self.n_heads = config.n_heads
        self.d_k = config.d_k

        # One full-width projection per role, carved into heads in forward().
        self.W_q = nn.Linear(config.d_model, config.d_model, bias=False)
        self.W_k = nn.Linear(config.d_model, config.d_model, bias=False)
        self.W_v = nn.Linear(config.d_model, config.d_model, bias=False)

        self.W_o = nn.Linear(config.d_model, config.d_model)

        mask = torch.tril(
            torch.ones(config.max_seq_len, config.max_seq_len, dtype=torch.bool)
        )

        self.register_buffer("mask", mask, persistent=False)

    def forward(self, x):

        batch_size, seq_len, d_model = x.shape

        q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        scores = q @ k.transpose(-2, -1)
        scores = scores / self.d_k**0.5

        scores = scores.masked_fill(~mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        out = weights @ v

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        return self.W_o(out), weights
        
        
